# 05 — Cas d'étude : analyse de consommation énergétique

**Module 1.1 — Premiers pas avec Claude Code**

---

Ce notebook est un cas d'étude complet : exploration, nettoyage, visualisation et interprétation métier d'un fichier CSV de consommation énergétique. L'objectif est de produire des insights actionnables en 40 minutes — un travail qui prendrait 2 heures dans un tableur.

**Durée estimée :** 40 minutes  
**Données :** `../data/consommation_energie.csv`  
**Prérequis :** Notebooks 01 à 04

---

## 1. Le contexte

**Scenario :** Responsable énergie dans un groupe industriel. Un comité de direction se tient demain matin. La direction demande un point sur la consommation des 2 dernières années, avec identification des postes prioritaires et des tendances.

Le service informatique a exporté un CSV depuis le système de Gestion Technique du Bâtiment (GTB). Le fichier n'a jamais été ouvert — on ne sait pas encore dans quel état il est.

**Ce qu'on doit produire :**
1. Un état des données (fiabilité, anomalies)
2. Quatre graphiques exploitables
3. Trois insights actionnables pour le comité

**Les 4 sites :**
- Usine_A et Usine_B : production industrielle
- Bureau_Central : tertiaire
- Entrepot_Nord : logistique

**Plan de travail :**

| Phase | Contenu | Durée |
|-------|---------|-------|
| Phase 1 | Exploration | 15 min |
| Phase 2 | Nettoyage | 10 min |
| Phase 3 | Visualisation (4 graphiques) | 15 min |
| Phase 4 | Insights et recommandations | — |

---

## Phase 1 : Exploration — 15 minutes

Avant d'écrire la moindre ligne d'analyse, il faut comprendre ce que contient le fichier : structure, volume, qualité. Cette phase ne produit aucun graphique — elle produit de la connaissance.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chargement du fichier brut
df_brut = pd.read_csv('../data/consommation_energie.csv')

print(f"Dimensions : {df_brut.shape[0]} lignes × {df_brut.shape[1]} colonnes")
print(f"\nColonnes : {df_brut.columns.tolist()}")
print(f"\nPériode couverte : {df_brut['date'].min()} → {df_brut['date'].max()}")

In [ ]:
# Aperçu des premières lignes
# head() donne une vue représentative de la structure réelle
df_brut.head(10)

In [ ]:
# Types de données et valeurs non nulles
# info() est le diagnostic de base — il révèle les problèmes de type
df_brut.info()

In [ ]:
# Statistiques descriptives sur les colonnes numériques
# La valeur min négative de kwh est déjà visible ici
df_brut.describe()

In [ ]:
# Inventaire des valeurs manquantes
nan_par_colonne = df_brut.isna().sum()
pct_manquant = (df_brut.isna().sum() / len(df_brut) * 100).round(2)

rapport_nan = pd.DataFrame({
    'Valeurs manquantes': nan_par_colonne,
    '% du total': pct_manquant
})
print("Valeurs manquantes par colonne :")
print(rapport_nan)
print(f"\nTotal lignes avec au moins un NaN : {df_brut.isna().any(axis=1).sum()}")

In [ ]:
# Valeurs négatives dans kwh
# Une consommation négative est physiquement impossible — c'est une erreur de capteur
negatifs = df_brut[df_brut['kwh'] < 0]
print(f"Lignes avec kwh négatif : {len(negatifs)}")
print()
print(negatifs[['date', 'batiment', 'zone', 'kwh']])

In [ ]:
# Doublons
n_doublons = df_brut.duplicated().sum()
print(f"Lignes en doublon : {n_doublons}")

if n_doublons > 0:
    print("\nExemple de doublons :")
    masque_doublon = df_brut.duplicated(keep=False)
    print(df_brut[masque_doublon].sort_values(['date', 'batiment', 'zone']).head(6))

In [ ]:
# Valeurs uniques par colonne catégorielle
# Permet de détecter les fautes de frappe (ex: 'Usine_a' vs 'Usine_A')
for col in ['batiment', 'zone', 'type_energie']:
    valeurs = sorted(df_brut[col].dropna().unique())
    print(f"\n{col} ({len(valeurs)} valeurs uniques) :")
    for v in valeurs:
        n = (df_brut[col] == v).sum()
        print(f"  {v} — {n} lignes")

In [ ]:
# Synthèse des anomalies trouvées
print("=" * 50)
print("BILAN EXPLORATION")
print("=" * 50)
print(f"Volume          : {len(df_brut)} lignes")
print(f"Période         : 2023-01-01 → 2024-12-31 (2 ans)")
print(f"Sites couverts  : {df_brut['batiment'].nunique()} bâtiments")
print()
print("Anomalies identifiées :")
print(f"  - Doublons     : {df_brut.duplicated().sum()} lignes")
print(f"  - kwh négatifs : {(df_brut['kwh'] < 0).sum()} lignes")
print(f"  - NaN (kwh)    : {df_brut['kwh'].isna().sum()} lignes")
print()
print("Décisions de nettoyage :")
print("  1. Supprimer les doublons exacts")
print("  2. Remplacer les kwh négatifs par NaN (erreur capteur)")
print("  3. Compléter les NaN par la moyenne du groupe (batiment × mois)")

---

## Phase 2 : Nettoyage — 10 minutes

Trois opérations dans l'ordre de leur impact : supprimer les doublons (perte de volume), neutraliser les valeurs impossibles (négatifs → NaN), puis imputer les NaN par une valeur contextualisée.

In [ ]:
# On travaille sur une copie — le fichier brut reste intact
df = df_brut.copy()

# Conversion de la colonne date en type datetime
# Indispensable pour les groupements par mois et les graphiques temporels
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

print(f"Avant nettoyage : {len(df)} lignes")

In [ ]:
# Étape 1 : suppression des doublons exacts
df = df.drop_duplicates()
print(f"Après suppression doublons : {len(df)} lignes")

In [ ]:
# Étape 2 : remplacer les kwh négatifs par NaN
# On ne supprime pas ces lignes — on neutralise uniquement la valeur aberrante
# L'information contextuelle (batiment, zone, date) reste utile pour l'imputation
masque_negatif = df['kwh'] < 0
df.loc[masque_negatif, 'kwh'] = None

print(f"Valeurs négatives remplacées par NaN : {masque_negatif.sum()}")
print(f"Total NaN dans kwh après cette étape : {df['kwh'].isna().sum()}")

In [ ]:
# Étape 3 : imputation des NaN par la moyenne du groupe batiment × mois
# La logique : un relevé manquant est remplacé par la consommation moyenne
# observée pour ce même bâtiment, ce même mois — ce qui préserve la saisonnalité

# transform() applique une fonction groupe par groupe et retourne une série
# de même index que le DataFrame d'origine — toutes les colonnes sont conservées
df['mois'] = df['date'].dt.month
df['kwh'] = df.groupby(['batiment', 'mois'])['kwh'].transform(
    lambda x: x.fillna(x.mean())
)

# NaN résiduels : si tout un groupe n'a aucune valeur, on ne peut pas imputer
nan_residuels = df['kwh'].isna().sum()
print(f"NaN après imputation : {nan_residuels}")
print(f"Lignes finales      : {len(df)}")

In [ ]:
# Sauvegarde du fichier nettoyé
chemin_propre = '../data/consommation_energie_propre.csv'
df.drop(columns=['mois']).to_csv(chemin_propre, index=False)
print(f"Fichier sauvegardé : {chemin_propre}")

In [ ]:
# Tableau comparatif avant / après nettoyage
comparaison = pd.DataFrame({
    'Indicateur': [
        'Nombre de lignes',
        'Valeurs NaN (kwh)',
        'Valeurs négatives (kwh)',
        'Doublons',
        'kwh moyen',
        'kwh min'
    ],
    'Avant nettoyage': [
        len(df_brut),
        df_brut['kwh'].isna().sum(),
        (df_brut['kwh'] < 0).sum(),
        df_brut.duplicated().sum(),
        round(df_brut['kwh'].mean(), 1),
        round(df_brut['kwh'].min(), 1)
    ],
    'Après nettoyage': [
        len(df),
        df['kwh'].isna().sum(),
        (df['kwh'] < 0).sum(),
        0,
        round(df['kwh'].mean(), 1),
        round(df['kwh'].min(), 1)
    ]
})

comparaison

---

## Phase 3 : Visualisation — 15 minutes

Quatre graphiques, quatre questions métier distinctes. Chaque graphique est conçu pour répondre à une question précise — pas pour être exhaustif.

### Graphique 1 — Consommation totale par bâtiment

**Question :** Quel bâtiment consomme le plus ? Quelle est la répartition électricité / gaz ?

**Type :** Barres groupées par type d'énergie

In [ ]:
# Agrégation : somme totale par bâtiment et type d'énergie
conso_batiment = (
    df
    .groupby(['batiment', 'type_energie'])['kwh']
    .sum()
    .reset_index()
)

# Conversion en MWh pour une lecture plus lisible (1 MWh = 1000 kWh)
conso_batiment['mwh'] = (conso_batiment['kwh'] / 1000).round(1)

fig1 = px.bar(
    conso_batiment,
    x='batiment',
    y='mwh',
    color='type_energie',
    barmode='group',
    title='Consommation totale par bâtiment (2023–2024)',
    labels={
        'batiment': 'Bâtiment',
        'mwh': 'Consommation (MWh)',
        'type_energie': 'Type d\'énergie'
    },
    color_discrete_map={
        'electricite': '#1f77b4',
        'gaz': '#ff7f0e'
    }
)

fig1.update_layout(
    xaxis_tickangle=0,
    legend_title_text='Énergie'
)

fig1.show()

**Lecture du graphique 1**

Les deux usines (Usine_A et Usine_B) dominent la consommation totale, avec une forte proportion de gaz liée aux processus industriels. Le Bureau_Central et l'Entrepot_Nord ont des profils très différents : quasi-exclusivement électrique, avec des volumes bien plus faibles.

**Insight métier :** Toute action de réduction doit cibler en priorité les deux usines. Une réduction de 10% sur Usine_A équivaut à diviser par plus de deux la consommation totale du Bureau_Central.

### Graphique 2 — Évolution mensuelle de la consommation

**Question :** Y a-t-il une saisonnalité ? Des mois anormalement élevés ou faibles ?

**Type :** Courbes par bâtiment, axe temporel

In [ ]:
# Agrégation mensuelle — toutes énergies confondues par bâtiment
df['annee_mois'] = df['date'].dt.to_period('M').dt.to_timestamp()

conso_mensuelle = (
    df
    .groupby(['annee_mois', 'batiment'])['kwh']
    .sum()
    .reset_index()
)

conso_mensuelle['mwh'] = (conso_mensuelle['kwh'] / 1000).round(1)

fig2 = px.line(
    conso_mensuelle,
    x='annee_mois',
    y='mwh',
    color='batiment',
    title='Évolution mensuelle de la consommation par bâtiment (2023–2024)',
    labels={
        'annee_mois': 'Mois',
        'mwh': 'Consommation (MWh)',
        'batiment': 'Bâtiment'
    },
    markers=True
)

fig2.update_layout(
    xaxis_title='',
    legend_title_text='Bâtiment'
)

fig2.show()

**Lecture du graphique 2**

La courbe révèle un pattern saisonnier : les consommations sont plus élevées en hiver (janvier–février, novembre–décembre) qu'en été. Ce pattern est attendu pour le chauffage au gaz, mais il est aussi visible sur l'électricité — ce qui peut indiquer une part de climatisation ou un effet de production.

**Insight métier :** La saisonnalité est prévisible. Une baseline mensuelle permettrait de distinguer les variations normales des anomalies réelles. Sans baseline, on ne peut pas savoir si un pic de janvier est "normal" ou non.

### Graphique 3 — Heatmap : zones × mois

**Question :** Quelles zones consomment le plus, et à quel moment de l'année ?

**Type :** Heatmap (matrice zones × mois)

In [ ]:
# Pivot table : zones en lignes, mois en colonnes, somme kwh
df['num_mois'] = df['date'].dt.month
df['nom_mois'] = df['date'].dt.strftime('%b')

heatmap_data = (
    df
    .groupby(['zone', 'num_mois'])['kwh']
    .sum()
    .reset_index()
)

# Pivot pour obtenir la matrice zones × mois
pivot = heatmap_data.pivot(index='zone', columns='num_mois', values='kwh')

# Labels de mois lisibles
noms_mois = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
             'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']

fig3 = go.Figure(
    data=go.Heatmap(
        z=pivot.values / 1000,  # conversion en MWh
        x=noms_mois,
        y=pivot.index.tolist(),
        colorscale='YlOrRd',
        colorbar=dict(title='MWh')
    )
)

fig3.update_layout(
    title='Consommation par zone et par mois — tous bâtiments (2023–2024, MWh)',
    xaxis_title='Mois',
    yaxis_title='Zone'
)

fig3.show()

**Lecture du graphique 3**

La zone Production concentre la majorité de la consommation — sur tous les mois. Les zones Éclairage et Climatisation ont des profils saisonniers distincts et inversés : l'éclairage consomme plus en hiver (nuits longues), la climatisation en été. Les Bureaux restent relativement stables.

**Insight métier :** La Production est le hotspot principal et mérite une instrumentation plus fine. La Climatisation présente un pic estival potentiellement optimisable (régulation de consigne, horaires).

### Graphique 4 — Température extérieure simulée vs consommation de gaz

**Question :** La consommation de gaz est-elle corrélée à la température extérieure ?

**Note :** Le dataset ne contient pas de colonne température. On génère ici une température mensuelle moyenne représentative d'un site en France métropolitaine. Cette démarche est documentée dans le graphique — en production, on utiliserait les données réelles de météo.

**Type :** Nuage de points avec droite de tendance

In [ ]:
import numpy as np

# Températures moyennes mensuelles en °C (France, zone tempérée)
# Source type : données Météo-France pour une station de référence
temp_mensuelle = {
    1: 4.2,   # Janvier
    2: 5.1,   # Février
    3: 8.9,   # Mars
    4: 12.3,  # Avril
    5: 16.1,  # Mai
    6: 19.8,  # Juin
    7: 22.4,  # Juillet
    8: 22.1,  # Août
    9: 17.9,  # Septembre
    10: 13.2, # Octobre
    11: 7.8,  # Novembre
    12: 4.9   # Décembre
}

# Sélection des relevés de gaz uniquement
df_gaz = df[df['type_energie'] == 'gaz'].copy()
df_gaz['num_mois'] = df_gaz['date'].dt.month
df_gaz['temperature_ext'] = df_gaz['num_mois'].map(temp_mensuelle)

# Agrégation mensuelle pour avoir un point par mois
scatter_data = (
    df_gaz
    .groupby(['annee_mois', 'temperature_ext', 'batiment'])['kwh']
    .sum()
    .reset_index()
)
scatter_data['mwh'] = (scatter_data['kwh'] / 1000).round(2)

fig4 = px.scatter(
    scatter_data,
    x='temperature_ext',
    y='mwh',
    color='batiment',
    trendline='ols',
    title='Corrélation température extérieure — consommation de gaz',
    labels={
        'temperature_ext': 'Température extérieure moyenne (°C)',
        'mwh': 'Consommation de gaz (MWh)',
        'batiment': 'Bâtiment'
    }
)

fig4.update_layout(
    legend_title_text='Bâtiment'
)

# Annotation pour documenter la source de la température
fig4.add_annotation(
    text="Note : températures issues d'une référence météo mensuelle (données réelles à intégrer)",
    xref="paper", yref="paper",
    x=0, y=-0.15,
    showarrow=False,
    font=dict(size=10, color='gray')
)

fig4.show()

**Lecture du graphique 4**

La droite de tendance montre une corrélation négative entre température et consommation de gaz : plus il fait froid, plus la consommation de gaz est élevée. Ce résultat est cohérent avec l'utilisation du gaz pour le chauffage des espaces et des procédés.

**Insight métier :** La consommation de gaz est largement pilotée par la météo. Pour évaluer l'efficacité des actions d'économie, il faut normaliser par la température (Degré Jour Unifié — DJU). Comparer un hiver 2023 doux à un hiver 2024 rigoureux sans correction climatique fausserait toute conclusion.

---

## Phase 4 : Insights métier

Trois insights issus directement des graphiques, formulés pour un comité de direction.

In [ ]:
# Calculs de support pour les insights

# Consommation totale par bâtiment
total_par_bat = df.groupby('batiment')['kwh'].sum().sort_values(ascending=False)
total_global = total_par_bat.sum()

print("Répartition de la consommation totale :")
for bat, kwh in total_par_bat.items():
    pct = kwh / total_global * 100
    print(f"  {bat:<20} {kwh/1000:>8.0f} MWh  ({pct:.1f}%)")
print(f"  {'TOTAL':<20} {total_global/1000:>8.0f} MWh")

print()

# Saisonnalité gaz : ratio hiver / été
df_gaz_mois = df[df['type_energie'] == 'gaz'].copy()
df_gaz_mois['saison'] = df_gaz_mois['date'].dt.month.map(
    lambda m: 'Hiver (Nov-Mar)' if m in [11, 12, 1, 2, 3] else 'Été (Mai-Sep)'
)
conso_saison = df_gaz_mois.groupby('saison')['kwh'].sum()
ratio = conso_saison['Hiver (Nov-Mar)'] / conso_saison['Été (Mai-Sep)']

print(f"Consommation de gaz — ratio hiver/été : {ratio:.1f}x")
print(f"  Hiver : {conso_saison['Hiver (Nov-Mar)']/1000:.0f} MWh")
print(f"  Été   : {conso_saison['Été (Mai-Sep)']/1000:.0f} MWh")

### Insight 1 — Concentration du risque énergétique

Les deux usines représentent environ 80% de la consommation totale du groupe sur 2023–2024. Le Bureau_Central et l'Entrepot_Nord, bien que visibles dans les budgets, sont marginaux en volume.

**Recommandation pour le comité :** Concentrer les investissements en efficacité énergétique sur les deux usines. Un point de réduction sur Usine_A ou Usine_B vaut dix points sur les sites tertiaires. Toute politique "tous sites" risque de diluer les efforts sans résultat significatif.

### Insight 2 — Saisonnalité non corrigée

La consommation de gaz en hiver est entre 2 et 3 fois supérieure à la consommation estivale. Or, les objectifs de réduction sont actuellement définis en kWh bruts, sans correction climatique.

**Recommandation pour le comité :** Adopter le DJU (Degré Jour Unifié) comme indicateur de normalisation. Un tableau de bord comparant la consommation réelle à la consommation "théorique à météo équivalente" permettrait de distinguer les gains réels des effets climatiques. Cela change la lecture : un hiver 2024 plus doux que 2023 ne signifie pas une amélioration de l'efficacité.

### Insight 3 — Zone Production : instrumentation insuffisante

La zone Production concentre la majorité de la consommation sur tous les mois et tous les bâtiments. Pourtant, les données disponibles ne permettent pas de distinguer les sous-postes au sein de cette zone (lignes de production, équipements spécifiques).

**Recommandation pour le comité :** Investir dans une instrumentation fine de la zone Production (sous-comptage par ligne ou par équipement). Sans cette granularité, on sait qu'il y a un problème mais on ne peut pas identifier où agir. L'analyse des données brutes GTB montre que le système actuel agrège trop.

### Ce qui méritera une investigation complémentaire

- Les 10 relevés négatifs identifiés en phase 1 : sont-ils tous des erreurs capteur, ou certains correspondent-ils à une rétrocession d'énergie (solaire, cogénération) ?
- Les mois avec des valeurs imputées : l'imputation par moyenne de groupe est conservatrice — une analyse de sensibilité devrait vérifier son impact sur les indicateurs clés
- L'évolution 2023 → 2024 à iso-météo : requiert les données DJU, non disponibles dans le fichier actuel

---

## Bilan

### Temps de travail

| Phase | Durée réelle | Tâche principale |
|-------|-------------|------------------|
| Exploration | 15 min | Chargement, stats, inventaire des anomalies |
| Nettoyage | 10 min | 3 opérations documentées, fichier propre sauvegardé |
| Visualisation | 15 min | 4 graphiques interactifs |
| **Total** | **40 min** | |

### Ce qu'on a appris sur les données

- Le CSV exporté depuis GTB contenait des erreurs de qualité (2% de données à traiter)
- La saisonnalité est forte et prévisible — elle masque les tendances réelles
- La granularité actuelle (zone, pas équipement) limite la capacité d'action

### Ce qu'aurait pris la même analyse sous Excel

| Étape | Excel | Python + Pandas |
|-------|-------|----------------|
| Détection des doublons | Mise en forme conditionnelle, manuelle | 1 ligne de code |
| Remplacement des négatifs | Filtre + saisie manuelle | 2 lignes de code |
| Imputation par groupe | Formule MOYENNE.SI, colonne auxiliaire | 5 lignes de code |
| 4 graphiques | 4 onglets, configuration manuelle | 4 blocs de code réutilisables |
| Mise à jour si nouvelles données | Tout refaire | Relancer le notebook |
| **Total estimé** | **~2h** | **~40 min** |

L'écart se creuse à chaque nouveau mois de données : en Python, la mise à jour est quasi-immédiate. En Excel, elle nécessite de reprendre la même procédure manuelle.

### Connexion avec la suite

Ce cas d'étude a utilisé les bases de pandas (chargement, nettoyage, agrégation) et de Plotly (4 types de graphiques). Les prochains modules approfondissent :

- **Module 2** — Automatisation : transformer ce notebook en pipeline qui s'exécute chaque mois sans intervention
- **Module 3** — Dashboard Streamlit : exposer ces graphiques dans une interface web accessible à toute l'équipe
- **Module 4** — Machine Learning : prédire la consommation future à partir de la température et des historiques